Import Libraries


In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, Dropout, LayerNormalization
from sklearn.metrics import make_scorer, f1_score, accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix, log_loss, matthews_corrcoef
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit
from tensorflow.keras.utils import to_categorical


2024-01-04 13:12:48.765152: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2024-01-04 13:12:49.166190: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-01-04 13:12:49.166319: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-01-04 13:12:49.237869: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-01-04 13:12:49.400281: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2024-01-04 13:12:49.405305: I tensorflow/core/platform/cpu_feature_guard.cc:1

In [2]:
data_df = pd.read_csv("seattle-weather.csv")
data_df


,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,drizzle
1,2012-01-02,10.9,10.6,2.8,4.5,rain
2,2012-01-03,0.8,11.7,7.2,2.3,rain
3,2012-01-04,20.3,12.2,5.6,4.7,rain
4,2012-01-05,1.3,8.9,2.8,6.1,rain
...,...,...,...,...,...,...
1456,2015-12-27,8.6,4.4,1.7,2.9,rain
1457,2015-12-28,1.5,5.0,1.7,1.3,rain
1458,2015-12-29,0.0,7.2,0.6,2.6,fog
1459,2015-12-30,0.0,5.6,-1.0,3.4,sun


Label Encoding

In [3]:
label_encoder = preprocessing.LabelEncoder()
data_df["weather"] = label_encoder.fit_transform(data_df["weather"])
num_of_class = len(data_df["weather"].unique())
num_of_class
data_df

,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,0
1,2012-01-02,10.9,10.6,2.8,4.5,2
2,2012-01-03,0.8,11.7,7.2,2.3,2
3,2012-01-04,20.3,12.2,5.6,4.7,2
4,2012-01-05,1.3,8.9,2.8,6.1,2
...,...,...,...,...,...,...
1456,2015-12-27,8.6,4.4,1.7,2.9,2
1457,2015-12-28,1.5,5.0,1.7,1.3,2
1458,2015-12-29,0.0,7.2,0.6,2.6,1
1459,2015-12-30,0.0,5.6,-1.0,3.4,4


In [4]:
data_df.drop("date",axis=1,inplace=True)

In [5]:
X_df = data_df.drop("weather",axis=1)
y_df = data_df["weather"]
X_df.head(10)

,precipitation,temp_max,temp_min,wind
0,0.0,12.8,5.0,4.7
1,10.9,10.6,2.8,4.5
2,0.8,11.7,7.2,2.3
3,20.3,12.2,5.6,4.7
4,1.3,8.9,2.8,6.1
5,2.5,4.4,2.2,2.2
6,0.0,7.2,2.8,2.3
7,0.0,10.0,2.8,2.0
8,4.3,9.4,5.0,3.4
9,1.0,6.1,0.6,3.4


In [6]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_X = scaler.fit_transform(X_df)

scaled_X[:7]

array([[0.        , 0.38709677, 0.47637795, 0.47252747],
       [0.19499106, 0.32795699, 0.38976378, 0.45054945],
       [0.01431127, 0.35752688, 0.56299213, 0.20879121],
       [0.36314848, 0.37096774, 0.5       , 0.47252747],
       [0.02325581, 0.28225806, 0.38976378, 0.62637363],
       [0.04472272, 0.16129032, 0.36614173, 0.1978022 ],
       [0.        , 0.23655914, 0.38976378, 0.20879121]])

In [7]:
X, y = [], []
time_step=7

for i in range(len(scaled_X)-time_step):
    X.append(scaled_X[i:i + time_step])
    y.append(y_df[i + time_step])

X, y = np.array(X), np.array(y)
X[0]

array([[0.        , 0.38709677, 0.47637795, 0.47252747],
       [0.19499106, 0.32795699, 0.38976378, 0.45054945],
       [0.01431127, 0.35752688, 0.56299213, 0.20879121],
       [0.36314848, 0.37096774, 0.5       , 0.47252747],
       [0.02325581, 0.28225806, 0.38976378, 0.62637363],
       [0.04472272, 0.16129032, 0.36614173, 0.1978022 ],
       [0.        , 0.23655914, 0.38976378, 0.20879121]])

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
y_train_encoded = to_categorical(y_train, num_classes=num_of_class)

print(X_train.shape)
print(y_train_encoded.shape)

(1163, 7, 4)
(1163, 5)


Creating the RNN model

In [9]:
def create_model(units=50, dropout_rate=0.2, learning_rate=0.001):
    model = Sequential()
    model.add(SimpleRNN(units=units, activation="sigmoid", input_shape=(X_train.shape[1], X_train.shape[2])))  
    model.add(LayerNormalization())
    model.add(Dropout(dropout_rate))
    model.add(Dense(units=num_of_class, activation="softmax"))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss="categorical_crossentropy", metrics=['accuracy'])

    return model

In [10]:
param_grid = {
    'model__units': [10, 20, 25, 30],
    'model__dropout_rate': [0.15, 0.2],
    'model__learning_rate': [0.001, 0.01, 0.1],  
    'batch_size': [32, 64, 128],
    'epochs': [50, 100, 150],
}

tscv = TimeSeriesSplit(n_splits=5)

keras_wrapped_model = KerasClassifier(model=create_model, verbose=0)

scorer = make_scorer(f1_score, average='weighted')

random_search = RandomizedSearchCV(
    keras_wrapped_model, param_distributions=param_grid, scoring=scorer,  cv=tscv, n_iter=20
)

random_search.fit(X_train, y_train_encoded)

print("Best Hyperparameters:", random_search.best_params_)

2024-01-04 13:13:20.121044: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-01-04 13:13:20.124856: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
/home/arda/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1760: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_pr

/home/arda/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1760: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/home/arda/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1760: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/home/arda/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1760: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/

In [53]:
results = random_search.cv_results_

# Iterate over the results and print the parameter combination and its corresponding score
for i in range(len(results['mean_test_score'])):
    params = results['params'][i]
    score = results['mean_test_score'][i]
    print(f"Params: {params} - Score: {score}")

Params: {'model__units': 50, 'model__learning_rate': 0.01, 'model__dropout_rate': 0.2, 'epochs': 100, 'batch_size': 128} - Score: 0.5276846944681967
Params: {'model__units': 50, 'model__learning_rate': 0.1, 'model__dropout_rate': 0.15, 'epochs': 100, 'batch_size': 64} - Score: 0.22554145087784586
Params: {'model__units': 50, 'model__learning_rate': 0.1, 'model__dropout_rate': 0.15, 'epochs': 100, 'batch_size': 128} - Score: 0.22554145087784586
Params: {'model__units': 50, 'model__learning_rate': 0.1, 'model__dropout_rate': 0.2, 'epochs': 100, 'batch_size': 32} - Score: 0.29355530709079825
Params: {'model__units': 64, 'model__learning_rate': 0.001, 'model__dropout_rate': 0.2, 'epochs': 50, 'batch_size': 64} - Score: 0.5018093146075813
Params: {'model__units': 128, 'model__learning_rate': 0.001, 'model__dropout_rate': 0.15, 'epochs': 150, 'batch_size': 64} - Score: 0.5038254045035169
Params: {'model__units': 50, 'model__learning_rate': 0.1, 'model__dropout_rate': 0.15, 'epochs': 50, 'bat

In [55]:
best_params = random_search.best_params_

model = create_model(
    units=best_params['model__units'],
    dropout_rate=best_params['model__dropout_rate'],
    learning_rate=best_params['model__learning_rate']
)

model.fit(X_train, y_train_encoded, epochs=best_params['epochs'], batch_size=best_params['batch_size'])


Epoch 1/150
10/10 [==============================] - 3s 23ms/step - loss: 1.7129 - accuracy: 0.4428
Epoch 2/150
10/10 [==============================] - 0s 24ms/step - loss: 1.3957 - accuracy: 0.4325
Epoch 3/150
10/10 [==============================] - 0s 12ms/step - loss: 1.1622 - accuracy: 0.4901
Epoch 4/150
10/10 [==============================] - 0s 7ms/step - loss: 1.1618 - accuracy: 0.4497
Epoch 5/150
10/10 [==============================] - 0s 7ms/step - loss: 1.1494 - accuracy: 0.4695
Epoch 6/150
10/10 [==============================] - 0s 11ms/step - loss: 1.1168 - accuracy: 0.4815
Epoch 7/150
10/10 [==============================] - 0s 8ms/step - loss: 1.1161 - accuracy: 0.5133
Epoch 8/150
10/10 [==============================] - 0s 7ms/step - loss: 1.0915 - accuracy: 0.5236
Epoch 9/150
10/10 [==============================] - 0s 6ms/step - loss: 1.0873 - accuracy: 0.5331
Epoch 10/150
10/10 [==============================] - 0s 6ms/step - loss: 1.0841 - accuracy: 0.5297
Epoch

In [65]:
probabilities = model.predict(X_test)
pred = np.argmax(probabilities, axis=1)

# Correctly classified instances out of the total instances
accuracy = accuracy_score(y_test, pred)
print(f'Accuracy: {accuracy}')

# predicted positive observations to the total predicted positives.
precision = precision_score(y_test, pred, average='weighted')  
print(f'Precision: {precision}')

# correctly predicted positive observations to the all observations in the actual class.
recall = recall_score(y_test, pred, average='weighted') 
print(f'Recall: {recall}')

# harmonic mean of precision and recall
f1 = f1_score(y_test, pred, average='weighted')  
print(f'F1 Score: {f1}')

# Confusion Matrix
conf_matrix = confusion_matrix(y_test, pred)
print(f'Confusion Matrix:\n{conf_matrix}')

# correlation coefficient between the observed and predicted binary classifications. Range [-1, 1] 1 perfect
mcc = matthews_corrcoef(y_test, pred)
print(f'Matthews Correlation Coefficient: {mcc}')

y_test= to_categorical(y_test, num_classes=num_of_class)

# It measures how well a probability distribution predicted by the model [0,1] is better
logloss = log_loss(y_test, probabilities)
print(f'Log Loss: {logloss}')

10/10 [==============================] - 0s 4ms/step
Accuracy: 0.6597938144329897
Precision: 0.5960625722084996
Recall: 0.6597938144329897
F1 Score: 0.6123753835597769
Confusion Matrix:
[[  0   0   0   0   7]
 [  0   0   7   0  24]
 [  0   2  61   3  42]
 [  0   0   0   0   0]
 [  0   0  14   0 131]]
Matthews Correlation Coefficient: 0.40371105161575926
Log Loss: 0.9505179111297569


/home/arda/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/arda/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
